# Module 1 - Data generation
## Medcore Pharma | Demand Planning Intelligence

Objective: Generate a realistic synthetic dataset simulating 3 years of demand planning data for 50 SKUs across 3 markets (France, Germany, Benelux).

Problem statement: S&OP exists at MedCore but KPIs are poorly interpreted.
This dataset intentionally embeds real-world pathologies that we will seek to detect:
structural forecast bias, uncaptured seasonality, bullwhip effect, business outliers and noise.

Stack: Python — pandas, numpy, SQLite

In [3]:
# Imports

import pandas as pd     #data manipulation
import numpy as np      #maths
import sqlite3      #sql connection
import os   #files and folders manipulations
from datetime import datetime, timedelta #dates manipulation

# versions check
print(f"pandas : {pd.__version__}")
print(f"numpy : {np.__version__}")
print(f"sqlite3 : {sqlite3.version}")
print("All packages are ok")

pandas : 2.3.3
numpy : 2.3.5
sqlite3 : 2.6.0
All packages are ok


C:\Users\USER\AppData\Local\Temp\ipykernel_9604\1606286089.py:12: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(f"sqlite3 : {sqlite3.version}")


In [9]:
# prject parameters

# time  period
DATE_START = datetime(2022,2,3) #first monday of 2022
DATE_END = datetime(2024,12,30)  #last monday of 2024
NB_WEEKS = 156 #3 years x 52 weeks

#markets
MARKETS = ["France", "Germany", "Benelux"]

#product families
FAMILIES = {
    "Medical Devices"       : 20,  # 20 SKUs
    "Surgical Consumables"  : 20,  # 20 SKUs
    "Diagnostic Equipment"  : 10   # 10 SKUs
}

# Average unit cost per family (in euros)
UNIT_COST = {
    "Medical Devices"       : 850,
    "Surgical Consumables"  : 45,
    "Diagnostic Equipment"  : 2200
}

# Database path
DB_PATH = os.path.join("..", "data", "medcore.db")

print(f"Period      : {DATE_START.date()} -> {DATE_END.date()}")
print(f"Weeks       : {NB_WEEKS}")
print(f"Markets     : {MARKETS}")
print(f"Families    : {list(FAMILIES.keys())}")
print(f"Database    : {DB_PATH}")
print("Parameters OK.")

Period      : 2022-02-03 -> 2024-12-30
Weeks       : 156
Markets     : ['France', 'Germany', 'Benelux']
Families    : ['Medical Devices', 'Surgical Consumables', 'Diagnostic Equipment']
Database    : ..\data\medcore.db
Parameters OK.


In [10]:
# Product Master data Generation
# We generate here the referance table for all 50 SKUs  (20+20+10) with their family, market and unit cost.
#Each SKU has a family, a  market, a unit cost a nd a pathalogy flags

import random
random.seed(42) # to be sure that we get the same results each time we run the code

skus = []
sku_id = 1

for family, nb_skus in FAMILIES.items() :
    for i in range(nb_skus) :
        market = random.choice(MARKETS)
        base_cost = UNIT_COST[family]
        unit_cost = round(base_cost *random.uniform(0.8,1.2),2)

        has_bias = random.random() <0.25 # 25% of SKUs have structural bias
        has_seasonality = random.random() < 0.30 #30% have uncaptured seasonality
        has_bullwhip = random.random() < 0.10 #10% are bullwhip amplifiers
    
        skus.append({
            "SKU_ID" :  f"SKU-{sku_id:03d}",
            "Family" : family,
            "Market" : market,
            "Unit_Cost" : unit_cost,
            "has_bias" : int(has_bias),
            "has_seasonality" : int(has_seasonality),
            "has_bullwhip" : int(has_bullwhip)
        })
        sku_id += 1

df_products = pd.DataFrame(skus)

print(f"Total SKUs generated : {len(df_products)}")
print(f"\nSKUs with bias flag        : {df_products['has_bias'].sum()}")
print(f"SKUs with seasonality flag : {df_products['has_seasonality'].sum()}")
print(f"SKUs with bullwhip flag    : {df_products['has_bullwhip'].sum()}")
print(f"\nFirst 5 rows :")
df_products.head()

Total SKUs generated : 50

SKUs with bias flag        : 20
SKUs with seasonality flag : 18
SKUs with bullwhip flag    : 7

First 5 rows :


,SKU_ID,Family,Market,Unit_Cost,has_bias,has_seasonality,has_bullwhip
0,SKU-001,Medical Devices,Benelux,717.85,0,1,0
1,SKU-002,Medical Devices,France,910.08,0,1,0
2,SKU-003,Medical Devices,France,711.86,1,0,0
3,SKU-004,Medical Devices,Benelux,900.96,0,1,0
4,SKU-005,Medical Devices,France,937.99,1,0,0


In [11]:
# Database creation & Product master data  insertion
# We create the database and insert the product master data

# Step 1 — Create the data folder if it doesn't exist
os.makedirs(os.path.join("..", "data"), exist_ok=True)

# Step 2 — Connect to the database
# If medcore.db doesn't exist yet, SQLite creates it automatically
conn = sqlite3.connect(DB_PATH)
# Step 3 — Insert the product master data into a table named "product_master"
df_products.to_sql(
    name = "product_master",
    con = conn,
    if_exists = "replace",
    index = False
)

# Step 4 — Verify that the data has been inserted correctly by querying the table
df_check = pd.read_sql_query("SELECT * FROM product_master LIMIT 5", conn)

print(f"Database created at : {DB_PATH}")
print(f"Rows written        : {len(df_products)}")
print(f"\nVerification — first 5 rows read from database :")
df_check

Database created at : ..\data\medcore.db
Rows written        : 50

Verification — first 5 rows read from database :


,SKU_ID,Family,Market,Unit_Cost,has_bias,has_seasonality,has_bullwhip
0,SKU-001,Medical Devices,Benelux,717.85,0,1,0
1,SKU-002,Medical Devices,France,910.08,0,1,0
2,SKU-003,Medical Devices,France,711.86,1,0,0
3,SKU-004,Medical Devices,Benelux,900.96,0,1,0
4,SKU-005,Medical Devices,France,937.99,1,0,0
